# Anomaly Investigation: `bank_id == 777` cluster

Standalone notebook — reloads and prepares the data independently of `01_eda.ipynb`.

Profiling the 635-row subset with the same tools used on the full dataset in `01_eda.ipynb`, and comparing against global baselines rather than looking at it in isolation.


In [1]:
import pandas as pd

df = pd.read_csv('/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv')

In [2]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

## 6. Investigating the `bank_id == 777` cluster

Profiling the 635-row subset with the same tools used on the full dataset, and comparing against global baselines rather than looking at it in isolation.

In [3]:
check_777 = df[df['bank_id'] == 777]
check_777

,created_at,order_id,processed_at,order_type,user_id,ip_country,currency,amount,payment_method,order_payment_type,bin_country,bank_id,psp_id,has_refund,refunded_amount,is_secured,status,error_code
999365,2025-12-01 13:16:39,999366,2025-12-01 13:16:44,recurring,769549,CAN,CAD,41.10,card,rebill,USA,777,psp_gamma,False,0.0,False,fail,4.09
999366,2025-12-01 00:30:43,999367,2025-12-01 00:30:48,first,888226,USA,USD,9.99,googlepay,NaN,USA,777,psp_alpha,False,0.0,False,fail,4.09
999367,2025-12-01 18:04:56,999368,2025-12-01 18:05:01,recurring,704040,USA,USD,30.00,card,recurring,CAN,777,psp_gamma,False,0.0,False,fail,4.09
999368,2025-12-01 04:24:47,999369,2025-12-01 04:24:52,recurring,567238,MEX,MXN,740.00,card,recurring,MEX,777,psp_alpha,False,0.0,False,fail,4.09
999369,2025-12-01 11:00:05,999370,2025-12-01 11:00:10,first,945225,USA,USD,9.99,applepay,NaN,USA,777,psp_beta,False,0.0,False,fail,4.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,2025-12-01 21:27:19,999996,2025-12-01 21:27:24,recurring,956515,USA,USD,200.00,card,retry,USA,777,psp_alpha,False,0.0,False,fail,3.08
999996,2025-12-01 22:16:31,999997,2025-12-01 22:16:36,first,177169,CAN,CAD,1.37,card,NaN,CAN,777,psp_alpha,False,0.0,False,fail,4.09
999997,2025-12-01 01:52:32,999998,2025-12-01 01:52:37,first,740896,CAN,CAD,13.70,card,NaN,CAN,777,psp_beta,False,0.0,False,fail,4.09
999998,2025-12-01 17:38:42,999999,2025-12-01 17:38:47,recurring,246155,POL,PLN,160.80,card,recurring,DEU,777,psp_gamma,False,0.0,False,fail,4.09


In [4]:
print(check_777.nunique())

created_at            633
order_id              635
processed_at          633
order_type              2
user_id               635
ip_country              8
currency                7
amount                 88
payment_method          3
order_payment_type      4
bin_country             8
bank_id                 1
psp_id                  3
has_refund              1
refunded_amount         1
is_secured              2
status                  1
error_code              4
dtype: int64


In [5]:
print(check_777.describe())

                          created_at        order_id  \
count                            635      635.000000   
mean   2025-12-01 12:12:57.272441088   999683.000000   
min              2025-12-01 00:00:42   999366.000000   
25%       2025-12-01 06:00:53.500000   999524.500000   
50%              2025-12-01 12:06:46   999683.000000   
75%              2025-12-01 18:35:45   999841.500000   
max              2025-12-01 23:59:55  1000000.000000   
std                              NaN      183.452991   

                        processed_at        user_id       amount  bank_id  \
count                            635     635.000000   635.000000    635.0   
mean   2025-12-01 12:13:02.272441088  529282.642520   112.904504    777.0   
min              2025-12-01 00:00:47  101570.000000     0.780000    777.0   
25%       2025-12-01 06:00:58.500000  303089.500000     9.995000    777.0   
50%              2025-12-01 12:06:51  517291.000000    23.400000    777.0   
75%              2025-12-01 18:35

In [6]:
for col in check_777.columns:
    print(check_777[col].value_counts())
    print("\n")

created_at
2025-12-01 09:46:33    2
2025-12-01 07:41:04    2
2025-12-01 13:16:39    1
2025-12-01 14:10:35    1
2025-12-01 11:30:18    1
                      ..
2025-12-01 08:07:56    1
2025-12-01 04:50:25    1
2025-12-01 20:16:26    1
2025-12-01 22:41:24    1
2025-12-01 11:20:19    1
Name: count, Length: 633, dtype: int64


order_id
999366     1
999792     1
999785     1
999786     1
999787     1
          ..
999578     1
999579     1
999580     1
999581     1
1000000    1
Name: count, Length: 635, dtype: int64


processed_at
2025-12-01 09:46:38    2
2025-12-01 07:41:09    2
2025-12-01 13:16:44    1
2025-12-01 14:10:40    1
2025-12-01 11:30:23    1
                      ..
2025-12-01 08:08:01    1
2025-12-01 04:50:30    1
2025-12-01 20:16:31    1
2025-12-01 22:41:29    1
2025-12-01 11:20:24    1
Name: count, Length: 633, dtype: int64


order_type
recurring    389
first        246
Name: count, dtype: int64


user_id
769549    1
552296    1
852552    1
488102    1
177941    1
         .

**Ruled out** (match the global distribution, so not distinctive to this group):
- `is_secured` rate is ~4%, same as the whole dataset;
- the share of round amounts (20.0, 30.0, 5.0, ...) is common globally too;
- geography (`ip_country`/`bin_country`, USA-dominated) roughly matches global proportions.

**Confirmed differences** (the actual fingerprint of this group):
- all 635 transactions were created on a **single calendar day** (hours vary, the date is fixed);
- all have `status == fail`, `has_refund == False`, `refunded_amount == 0.0`;
- `order_id` values are the **largest in the dataset** (the group sits at the physical end of the file);
- the `bank_id` format (777, three digits) does not match the pattern of every other bank.

Latency for this group is checked below.

In [7]:
duration = check_777['processed_at'] - check_777['created_at']
print(duration.nunique())
print(duration.describe())

1
count                635
mean     0 days 00:00:05
std      0 days 00:00:00
min      0 days 00:00:05
25%      0 days 00:00:05
50%      0 days 00:00:05
75%      0 days 00:00:05
max      0 days 00:00:05
dtype: object


**Strong evidence of artificial origin:** latency (processed_at - created_at) is exactly **5 seconds for all 635 rows, with zero variation**. Real network/banking processing cannot behave like this - it is a statistical signature of a synthetically generated or inserted group.